# Orivox - Separador de Pistas (gratuito)

Separa sua musica em pistas individuais e ainda gera, de brinde, os dados que o **Orivox** usa para transcrever e ensaiar com precisao: mapa de batidas, cifra dos acordes, e uma partitura de referencia da voz.

### O que voce recebe ao final (tudo num ZIP)
**Pistas:** Voz principal, Backing vocals, Bateria, Baixo, Guitarra, Piano, Outros, e a Voz completa de referencia.

**Extras para o Orivox:**
- Mapa de batidas (`_batidas.json`) - para o metronomo inteligente seguir a musica.
- Cifra dos acordes com tempo (`_acordes.musicxml` e `_acordes.txt`) - para o filtro harmonico.
- Partitura de referencia da voz (`_voz.mid` e `_voz.musicxml`) - para comparar e corrigir a transcricao do Orivox.
- Tom, andamento e afinacao da musica (dentro do `_batidas.json`).

### Como usar (passo a passo)
1. No menu acima: **Ambiente de execucao > Alterar tipo de ambiente de execucao**, escolha **T4 GPU** e salve.
2. Rode as celulas na ordem, clicando no play de cada uma.
3. Na celula 2, envie sua musica quando o botao de upload aparecer.
4. Ao final, os arquivos baixam sozinhos. Importe as pistas e os extras no Orivox.

*Tempo total para uma musica de 3 a 4 minutos: por volta de 4 a 7 minutos com GPU (a primeira execucao demora um pouco mais porque baixa os modelos).*

## Celula 1 - Instalar as ferramentas (rode uma vez por sessao)

In [ ]:
%%capture
!pip install -U demucs
!pip install "audio-separator[cpu]"
!pip install basic-pitch
!pip install music21
# Demucs: separa as pistas. audio-separator: divide voz principal x backing vocals.
# basic-pitch (Spotify): transcreve a voz isolada em MIDI. music21: monta a cifra em MusicXML.
# O onnxruntime em versao CPU evita conflitos de CUDA no Colab.


In [ ]:
print('Ferramentas instaladas! Pode seguir para a celula 2.')

## Celula 2 - Enviar sua musica (MP3, WAV, M4A...)

In [ ]:
from google.colab import files
import os, shutil

# Aviso amigavel se a GPU nao estiver ligada (nao impede, so avisa).
try:
    import torch
    if torch.cuda.is_available():
        print('GPU ligada:', torch.cuda.get_device_name(0), '- otimo, vai ser rapido.')
    else:
        print('ATENCAO: a GPU nao esta ligada. Vai funcionar, mas bem mais devagar.')
        print('Para ligar: menu Ambiente de execucao > Alterar tipo de ambiente > T4 GPU > Salvar. Depois rode de novo.')
except Exception:
    pass

if os.path.exists('/content/entrada'):
    shutil.rmtree('/content/entrada')
os.makedirs('/content/entrada', exist_ok=True)

print('\nClique em "Escolher arquivos" e selecione sua musica:')
enviados = files.upload()

for nome in enviados:
    shutil.move(nome, f'/content/entrada/{nome}')
    print(f'Recebido: {nome}')
arquivo = list(enviados.keys())[0]

## Celula 3 - Separar os instrumentos e a voz (Demucs, 6 pistas)

Gera 6 pistas: voz (todas juntas), bateria, baixo, guitarra, piano e outros. E o padrao do Orivox.

In [ ]:
MODO = 'htdemucs_6s'

import subprocess, os, shutil
if os.path.exists('/content/saida'):
    shutil.rmtree('/content/saida')
entrada = f'/content/entrada/{arquivo}'
print(f'Separando "{arquivo}" em 6 pistas... aguarde (pode levar alguns minutos).')
resultado = subprocess.run(
    ['demucs', '-n', MODO, '--mp3', '--mp3-bitrate', '320', '-o', '/content/saida', entrada],
    capture_output=True, text=True
)
if resultado.returncode == 0:
    print('Pronto! Instrumentos e voz separados.')
else:
    print('Algo deu errado na separacao:')
    print((resultado.stderr or '')[-1500:])

## Celula 4 - Dividir a voz em PRINCIPAL e BACKING VOCALS

Pega a pista de voz gerada acima e aplica um modelo especializado (tipo karaoke) que isola a voz principal dos vocais de apoio.

In [ ]:
import glob, os, shutil
from audio_separator.separator import Separator

candidatos = glob.glob('/content/saida/*/*/vocals.*')
if not candidatos:
    raise SystemExit('Pista de voz nao encontrada. Rode a celula 3 primeiro.')
voz_completa = candidatos[0]
print('Pista de voz localizada:', voz_completa)

if os.path.exists('/content/vozes'):
    shutil.rmtree('/content/vozes')
os.makedirs('/content/vozes', exist_ok=True)

print('Separando voz principal dos backing vocals... aguarde.')
sep = Separator(output_dir='/content/vozes', output_format='mp3')
sep.load_model(model_filename='5_HP-Karaoke-UVR.pth')
saidas = sep.separate(voz_completa)

print('Pronto! Arquivos de voz gerados:')
for s in saidas:
    print('  -', os.path.basename(s))

## Celula 5 - Partitura de referencia da voz (para corrigir a transcricao do Orivox)

Transcreve a **voz principal ja isolada** em MIDI usando o basic-pitch (do Spotify). Como a voz vem limpa, o resultado costuma ser bem melhor que transcrever a musica inteira.

No Orivox, importe o `_voz.mid` pelo botao **Importar referencia (comparar)** da Partitura: o Orivox mostra onde a transcricao dele bate com esta referencia e onde diverge, para voce corrigir. Tambem sai um `_voz.musicxml` para abrir direto no MuseScore/Encore.

*E uma referencia automatica: ajuda muito, mas sempre vale uma conferida final do seu ouvido.*

In [ ]:
import glob, os

# Usa a VOZ PRINCIPAL isolada (melhor fonte). Se nao houver, usa a voz completa.
voz_ref = None
for padrao in ['/content/vozes/*(Vocals)*', '/content/saida/*/*/vocals.*']:
    achados = glob.glob(padrao)
    if achados:
        voz_ref = achados[0]; break
if not voz_ref:
    raise SystemExit('Voz nao encontrada. Rode as celulas 3 e 4 primeiro.')
print('Transcrevendo a voz:', os.path.basename(voz_ref))

base = os.path.splitext(arquivo)[0]
os.makedirs('/content/extras', exist_ok=True)

# basic-pitch gera um MIDI a partir do audio da voz.
from basic_pitch.inference import predict_and_save
from basic_pitch import ICASSP_2022_MODEL_PATH
print('Analisando com basic-pitch... 1 a 2 minutos.')
predict_and_save(
    [voz_ref], '/content/extras',
    save_midi=True, sonify_midi=False, save_model_outputs=False, save_notes=False,
    model_or_model_path=ICASSP_2022_MODEL_PATH
)

# Renomeia o MIDI gerado para um nome claro.
midis = [m for m in glob.glob('/content/extras/*.mid*')]
voz_midi = f'/content/extras/{base}_voz.mid'
if midis:
    os.replace(midis[0], voz_midi)
    print('MIDI de referencia gerado:', os.path.basename(voz_midi))

# Converte o MIDI em MusicXML (partitura) com o music21, para abrir no MuseScore/Encore.
try:
    from music21 import converter
    part = converter.parse(voz_midi)
    voz_xml = f'/content/extras/{base}_voz.musicxml'
    part.write('musicxml', fp=voz_xml)
    print('Partitura MusicXML gerada:', os.path.basename(voz_xml))
except Exception as e:
    print('Nao consegui gerar o MusicXML da voz (o MIDI ja serve para comparar):', e)

## Celula 6 - Cifra dos acordes com tempo (para o filtro harmonico do Orivox)

Detecta a sequencia de acordes e o instante de cada mudanca. Gera um `_acordes.musicxml` (que encaixa no **filtro harmonico** do Orivox, mesmo formato do Chordify/Encore) e um `_acordes.txt` para voce ler com o olho.

*Funciona melhor em musicas com harmonia clara. Sempre vale conferir os acordes com o seu ouvido.*

In [ ]:
import glob, os, subprocess
import numpy as np

# Prefere analisar a musica COMPLETA (a harmonia inteira), nao so uma pista.
caminho = glob.glob('/content/entrada/*')[0]
base = os.path.splitext(os.path.basename(caminho))[0]
os.makedirs('/content/extras', exist_ok=True)

# Detecta os acordes com o Chordino (vamp) via a biblioteca 'chord-extractor'.
acordes = []  # lista de (tempo_seg, nome_acorde)
try:
    subprocess.run(['pip','install','-q','chord-extractor'], check=True)
    from chord_extractor.extractors import Chordino
    print('Detectando acordes com Chordino... 1 a 3 minutos.')
    ch = Chordino()
    res = ch.extract(caminho)
    for c in res:
        nome = getattr(c, 'chord', None)
        t = getattr(c, 'timestamp', None)
        if nome and t is not None and nome != 'N':  # 'N' = sem acorde
            acordes.append((float(t), str(nome)))
    print(f'Chordino encontrou {len(acordes)} mudancas de acorde.')
except Exception as e:
    print('Chordino indisponivel, tentando o metodo alternativo (autochord)...')
    try:
        subprocess.run(['pip','install','-q','autochord'], check=True)
        import autochord
        res = autochord.recognize(caminho)
        for (ini, fim, nome) in res:
            if nome and nome != 'N':
                acordes.append((float(ini), str(nome)))
        print(f'autochord encontrou {len(acordes)} mudancas de acorde.')
    except Exception as e2:
        print('Nao foi possivel detectar acordes nesta sessao:', e2)

# Funde acordes repetidos seguidos (mesmo nome em blocos consecutivos).
fundidos = []
for t, nome in sorted(acordes):
    if fundidos and fundidos[-1][1] == nome:
        continue
    fundidos.append((t, nome))
acordes = fundidos


In [ ]:
# 1) TXT legivel: 0:00 Cm | 0:04 G | ...
def mmss(seg):
    m = int(seg // 60); s = int(seg % 60); return f'{m}:{s:02d}'
txt = f'Cifra de {base}\n(gerada pelo Separador Orivox - confira com o ouvido)\n\n'
if acordes:
    txt += '  '.join(f'{mmss(t)} {nome}' for t, nome in acordes)
else:
    txt += '(nenhum acorde detectado nesta sessao)'
acordes_txt = f'/content/extras/{base}_acordes.txt'
open(acordes_txt, 'w').write(txt)
print('Cifra em texto:', os.path.basename(acordes_txt))

# 2) MusicXML de acordes com tempo, para o filtro harmonico do Orivox.
#    Cada acorde vira um compasso/harmonia com o instante certo (via duracao).
acordes_xml = f'/content/extras/{base}_acordes.musicxml'
try:
    from music21 import stream, harmony, tempo, meter, duration as m21dur
    sc = stream.Score(); part = stream.Part()
    if acordes:
        # descobre um BPM aproximado pelo espacamento medio dos acordes nao ajuda;
        # usamos tempo real: cada bloco dura ate o proximo acorde (em segundos),
        # convertido em batidas assumindo 120 BPM de grade (o Orivox le pelo tempo).
        part.append(tempo.MetronomeMark(number=120))
        part.append(meter.TimeSignature('4/4'))
        fim_musica = acordes[-1][0] + 4
        limites = [t for t, _ in acordes] + [fim_musica]
        for i, (t, nome) in enumerate(acordes):
            dur_seg = max(0.5, limites[i+1] - t)
            ql = dur_seg * (120/60.0)  # segundos -> quarter lengths a 120 BPM
            try:
                cs = harmony.ChordSymbol(nome.replace(':', ''))
            except Exception:
                cs = harmony.ChordSymbol('C')
                cs.figure = nome
            cs.duration = m21dur.Duration(ql)
            part.append(cs)
    sc.append(part)
    sc.write('musicxml', fp=acordes_xml)
    print('Cifra MusicXML (filtro harmonico):', os.path.basename(acordes_xml))
except Exception as e:
    print('Nao consegui gerar o MusicXML de acordes (o TXT ja serve de referencia):', e)

## Celula 7 - Mapa de batidas + tom + afinacao (para o metronomo do Orivox)

Gera o `_batidas.json` com cada batida e cada tempo forte da musica (por rede neural), mais o **tom** (ex.: Re maior), o **BPM** e a **afinacao de referencia** (se a musica esta em 440Hz ou desviada). Importe no metronomo inteligente do Orivox.

In [ ]:
import json, os, glob
import numpy as np

caminho = glob.glob('/content/entrada/*')[0]
base = os.path.splitext(os.path.basename(caminho))[0]
os.makedirs('/content/extras', exist_ok=True)

# ---- batidas e tempos fortes (madmom, com fallback no librosa) ----
beats, numeros, origem = None, None, None
try:
    import subprocess
    subprocess.run(['pip','install','-q','madmom'], check=True)
    from madmom.features.downbeats import RNNDownBeatProcessor, DBNDownBeatTrackingProcessor
    print('Analisando batidas com rede neural (madmom)... 1 a 3 minutos.')
    act = RNNDownBeatProcessor()(caminho)
    proc = DBNDownBeatTrackingProcessor(beats_per_bar=[3,4], fps=100)
    res = proc(act)
    beats = res[:,0].tolist()
    numeros = [int(n) for n in res[:,1]]
    origem = 'madmom (rede neural)'
except Exception as e:
    print('madmom indisponivel, usando o detector classico (librosa)...')
    import librosa
    y, sr = librosa.load(caminho, sr=22050, mono=True)
    tempo_, frames = librosa.beat.beat_track(y=y, sr=sr, trim=False)
    beats = librosa.frames_to_time(frames, sr=sr).tolist()
    env = librosa.onset.onset_strength(y=y, sr=sr)
    forca = [float(env[min(f, len(env)-1)]) for f in frames]
    melhor_o, melhor_s = 0, -1
    for o in range(4):
        s = sum(forca[o::4])
        if s > melhor_s: melhor_s, melhor_o = s, o
    numeros = [((i - melhor_o) % 4) + 1 for i in range(len(beats))]
    origem = 'librosa (classico)'

iv = np.diff(beats)
bpm = round(float(60/np.median(iv)), 1) if len(iv) else 0


In [ ]:
# ---- tom/clave e afinacao de referencia (librosa) ----
tom, afinacao_hz = None, None
try:
    import librosa
    y, sr = librosa.load(caminho, sr=22050, mono=True)
    # Tom: perfil de croma comparado aos perfis maior/menor de Krumhansl.
    croma = librosa.feature.chroma_cqt(y=y, sr=sr).mean(axis=1)
    maj = np.array([6.35,2.23,3.48,2.33,4.38,4.09,2.52,5.19,2.39,3.66,2.29,2.88])
    minp = np.array([6.33,2.68,3.52,5.38,2.60,3.53,2.54,4.75,3.98,2.69,3.34,3.17])
    notas = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']
    melhor = (-1, None)
    for i in range(12):
        for perfil, modo in [(maj,'maior'),(minp,'menor')]:
            r = float(np.corrcoef(np.roll(croma, -i), perfil)[0,1])
            if r > melhor[0]: melhor = (r, f'{notas[i]} {modo}')
    tom = melhor[1]
    # Afinacao: desvio em cents em relacao ao 440Hz padrao.
    try:
        cents = float(np.mean(librosa.pitch_tuning(y, sr=sr)) * 100)
        afinacao_hz = round(440.0 * (2 ** (cents/1200.0)), 1)
    except Exception:
        afinacao_hz = 440.0
    print(f'Tom estimado: {tom} | Afinacao aprox.: {afinacao_hz} Hz')
except Exception as e:
    print('Nao consegui estimar tom/afinacao:', e)

# ---- monta o JSON final ----
mapa = {
    'origem': origem, 'bpm': bpm,
    'tom': tom, 'afinacao_hz': afinacao_hz,
    'beats': [round(b,4) for b in beats],
    'beat_numbers': numeros
}
saida = f'/content/extras/{base}_batidas.json'
json.dump(mapa, open(saida,'w'), ensure_ascii=False)
print(f'\nMapa gerado: {len(beats)} batidas, ~{bpm} BPM, tom {tom}, origem {origem}')

## Celula 8 - Reunir tudo e baixar o ZIP final

In [ ]:
import shutil, os, glob
from google.colab import files

try:
    arquivo
except NameError:
    arquivo = os.path.basename(glob.glob('/content/entrada/*')[0])
base = os.path.splitext(arquivo)[0]
final = f'/content/{base}_Orivox'
if os.path.exists(final):
    shutil.rmtree(final)
os.makedirs(final, exist_ok=True)

# 1) Pistas do Demucs (a voz completa entra como referencia).
nomes_pt = {'drums':'Bateria','bass':'Baixo','other':'Outros','guitar':'Guitarra','piano':'Piano'}
pastas = [p for p in glob.glob('/content/saida/*/*') if os.path.isdir(p)]
if pastas:
    for f in glob.glob(f'{pastas[0]}/*'):
        raiz = os.path.splitext(os.path.basename(f))[0]
        ext = os.path.splitext(f)[1]
        if raiz == 'vocals':
            shutil.copy(f, f'{final}/Vozes juntas (referencia) - {base}{ext}')
        elif raiz in nomes_pt:
            shutil.copy(f, f'{final}/{nomes_pt[raiz]} - {base}{ext}')

# 2) Voz principal e backing vocals.
for f in glob.glob('/content/vozes/*'):
    nome = os.path.basename(f); ext = os.path.splitext(f)[1]
    if '(Vocals)' in nome:
        shutil.copy(f, f'{final}/Voz principal - {base}{ext}')
    elif '(Instrumental)' in nome:
        shutil.copy(f, f'{final}/Backing vocals - {base}{ext}')

# 3) Extras (mapa, cifra, partitura de voz) numa subpasta clara.
extras_dir = f'{final}/Extras para o Orivox'
os.makedirs(extras_dir, exist_ok=True)
for f in glob.glob('/content/extras/*'):
    shutil.copy(f, f'{extras_dir}/{os.path.basename(f)}')

zipado = shutil.make_archive(f'/content/{base}_Orivox', 'zip', final)
print('Tudo pronto para o Orivox:')
for f in sorted(os.listdir(final)):
    print('  -', f)
if os.path.isdir(extras_dir):
    for f in sorted(os.listdir(extras_dir)):
        print('     . Extras/'+f)
files.download(zipado)
print('\nDownload iniciado! Descompacte e importe no Orivox:')
print(' - As pistas: no console multipista.')
print(' - _batidas.json: no metronomo inteligente.')
print(' - _acordes.musicxml: no filtro harmonico (Importar acordes).')
print(' - _voz.mid: na Partitura, botao Importar referencia (comparar).')

---
### Dicas de qualidade
- **Backing vocals com residuos:** e uma limitacao natural desse tipo de separacao e depende de como as vozes foram mixadas. Vale testar o modelo alternativo trocando, na celula 4, `5_HP-Karaoke-UVR.pth` por `UVR-BVE-4B_SN-44100-1.pth` e rodando as celulas 4 e 8 de novo.
- **Partitura da voz e cifra:** sao referencias automaticas. Ajudam muito, especialmente para comparar e corrigir a transcricao do Orivox, mas sempre vale a conferida final do seu ouvido.
- **Musicas sem bateria:** o mapa de batidas por rede neural (celula 7) ajuda o metronomo a seguir mesmo so com voz e violao.

*Orivox - Separador de Pistas - uso gratuito via Google Colab - Desenvolvido por Raphael Silveira*